<a href="https://colab.research.google.com/github/Kellia855/intro-to-ml_summative/blob/main/copd_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Estimating the Likelihood of a Reported COPD Diagnosis in Adults 50+

**Goal.** COPD is often underdiagnosed because many people never get a breathing test. This project compares traditional machine learning and deep learning models that estimate how likely an adult aged 50 or older is to report a physician diagnosis of COPD, using self-reported demographic, lifestyle and health-history answers. It is intended as a **screening aid** to help prioritise people for clinical testing, not a diagnostic tool.

**What the model predicts.** The target `copd` is 1 if the respondent reports ever being told by a doctor that they have COPD, emphysema or chronic bronchitis (`CHCCOPD3`) and 0 otherwise. The model learns from about 269,000 respondents' answers and returns a probability for a new person. It does not diagnose COPD or forecast future disease, and undiagnosed cases appear as negatives in the data.

**Data.** CDC Behavioral Risk Factor Surveillance System (BRFSS) 2023, open US federal data:

## 1. Setup

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
import shutil, zipfile, os
zip_path = "/content/drive/MyDrive/ml_project/LLCP2023XPT.zip"
with zipfile.ZipFile(zip_path) as z:
    z.extractall("/content/data")
# the file inside has a trailing space in its name, so find it and rename it
for f in os.listdir("/content/data"):
    if f.strip().upper() == "LLCP2023.XPT":
        os.rename(f"/content/data/{f}", "/content/data/LLCP2023.XPT")

In [3]:
import numpy as np
import pandas as pd

# ---- Configuration -------------------------------------------------------
DATA_PATH = "/content/data/LLCP2023.XPT"            # path to the raw XPT file (rename if the unzipped name has a trailing space)
OUT_PATH  = "/content/drive/MyDrive/ml_project/brfss_copd_50plus.parquet"
SEED      = 42
CHUNK     = 50_000                    # rows per chunk; the full file (350 columns) does not fit in free Colab RAM

pd.set_option("display.max_rows", 100)

## 2. Variables

Every name below was checked against the 2023 file. Variable meanings and value codes come from the 2023 codebook (`USCODE23_LLCP_021924.HTML`).

* **Target:** `CHCCOPD3` — ever told you have COPD, emphysema or chronic bronchitis (1 = yes, 2 = no, 7/9 = don't know/refused).
* **Cohort filter:** `_AGEG5YR` — 5-year age category; codes 7–13 are ages 50+, code 14 is don't know/refused.
* **Features:** grouped below. `POORHLTH` is deliberately left out (it is only asked conditionally and may partly reflect COPD symptoms), and race is excluded by default so that any use of it is a documented, justified experiment.

In [4]:
TARGET = "CHCCOPD3"
AGE    = "_AGEG5YR"

FEATURES = {
    "demographics": ["_AGE80", "SEXVAR", "MARITAL", "EMPLOY1", "EDUCA", "INCOME3", "VETERAN3", "_URBSTAT", "_STATE"],
    "smoking_substances": ["SMOKE100", "_SMOKER3", "SMOKDAY2", "LASTSMK2", "USENOW3", "ALCDAY4", "_DRNKWK2"],
    "health_status": ["GENHLTH", "PHYSHLTH", "MENTHLTH", "_BMI5", "_BMI5CAT", "EXERANY2", "DIFFWALK", "DIFFDRES", "DIFFALON"],
    "comorbidities": ["ASTHMA3", "CVDINFR4", "CVDCRHD4", "CVDSTRK3", "DIABETE4", "BPHIGH6", "TOLDHI3", "ADDEPEV3", "HAVARTH4", "CHCKDNY2"],
    "healthcare_use": ["CHECKUP1", "MEDCOST1", "FLUSHOT7", "PNEUVAC4"],
}
FEATURE_COLS = [c for cols in FEATURES.values() for c in cols]
KEEP = [TARGET, AGE] + FEATURE_COLS
print(len(FEATURE_COLS), "candidate features")

39 candidate features


## 3. Load the raw file in chunks

Only the columns we need are kept, and the **raw row count** is recorded while reading, because the assignment requires reporting the dataset size *before* any preprocessing.

In [5]:
reader = pd.read_sas(DATA_PATH, format="xport", chunksize=CHUNK, encoding="latin-1")
parts, n_raw, n_cols_raw = [], 0, None
for chunk in reader:
    n_cols_raw = chunk.shape[1]
    n_raw += len(chunk)
    parts.append(chunk[KEEP].copy())
raw = pd.concat(parts, ignore_index=True)
del parts

print(f"Raw file: {n_raw:,} rows x {n_cols_raw} columns")
print(f"Kept    : {raw.shape[0]:,} rows x {raw.shape[1]} columns")
raw.head()

Raw file: 433,323 rows x 350 columns
Kept    : 433,323 rows x 41 columns


,CHCCOPD3,_AGEG5YR,_AGE80,SEXVAR,MARITAL,EMPLOY1,EDUCA,INCOME3,VETERAN3,_URBSTAT,...,DIABETE4,BPHIGH6,TOLDHI3,ADDEPEV3,HAVARTH4,CHCKDNY2,CHECKUP1,MEDCOST1,FLUSHOT7,PNEUVAC4
0,2.0,13.0,80.0,2.0,1.0,7.0,5.0,99.0,2.0,1.0,...,1.0,1.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0
1,2.0,13.0,80.0,2.0,2.0,7.0,5.0,99.0,2.0,1.0,...,3.0,1.0,1.0,1.0,1.0,2.0,2.0,2.0,1.0,1.0
2,2.0,13.0,80.0,2.0,3.0,7.0,4.0,2.0,2.0,1.0,...,3.0,1.0,1.0,2.0,1.0,2.0,1.0,1.0,1.0,1.0
3,2.0,12.0,78.0,2.0,1.0,7.0,5.0,99.0,2.0,1.0,...,3.0,3.0,2.0,1.0,1.0,2.0,3.0,2.0,1.0,1.0
4,2.0,12.0,76.0,2.0,3.0,8.0,5.0,7.0,2.0,1.0,...,1.0,1.0,2.0,1.0,1.0,2.0,1.0,2.0,2.0,1.0


## 4. Define the cohort

1. Keep respondents with a valid target answer (1 = yes, 2 = no); drop "don't know", "refused" and blank.
2. Keep adults aged 50 and older (`_AGEG5YR` 7–13); drop code 14.

The funnel table documents how many rows survive each step.

In [6]:
funnel = [("Raw BRFSS 2023 records", len(raw))]

df = raw[raw[TARGET].isin([1, 2])].copy()
funnel.append(("Valid COPD answer (1 or 2)", len(df)))

df = df[df[AGE].between(7, 13)].copy()
funnel.append(("Adults 50+ (_AGEG5YR 7-13)", len(df)))

df["copd"] = (df[TARGET] == 1).astype(int)
funnel_df = pd.DataFrame(funnel, columns=["step", "rows"])
funnel_df["dropped"] = funnel_df["rows"].diff().fillna(0).astype(int)
display(funnel_df)

prev = df["copd"].mean()
print(f"COPD prevalence in cohort: {prev:.2%}  ({df['copd'].sum():,} positives, imbalance about 1:{(1-prev)/prev:.1f})")

,step,rows,dropped
0,Raw BRFSS 2023 records,433323,0
1,Valid COPD answer (1 or 2),431257,-2066
2,Adults 50+ (_AGEG5YR 7-13),269252,-162005


COPD prevalence in cohort: 10.56%  (28,432 positives, imbalance about 1:8.5)


## 5. Recode special values

BRFSS stores "don't know", "refused" and "none" as numeric codes that would be read as real values if left in place. Each rule below follows the 2023 codebook:

* Yes/no questions: 1 = yes, 2 = no, 7 and 9 = don't know/refused → missing.
* Days variables (`PHYSHLTH`, `MENTHLTH`): 88 = none → 0; 77 and 99 → missing.
* `ALCDAY4`: 101–199 = days per week, 201–299 = days per month, 888 = no drinks, 777/999 → missing. Converted to **drinking days per month**.
* `_DRNKWK2`: 99900 = missing; the stored value has two implied decimals, and a true zero is stored by SAS as a tiny denormal number (about 5e-79), which is snapped to exactly 0.
* `_BMI5`: two implied decimals (divide by 100).
* Ordinal/nominal categoricals: refusal codes → missing.
* Blank values on `SMOKDAY2`, `LASTSMK2` and `USENOW3` are *structural* (the question was skipped by design) and are kept as missing for now. Whether to impute, add indicator columns, or use an "not applicable" category is a modelling decision to document and compare.

In [7]:
def recode(d):
    d = d.copy()

    yes_no = ["SMOKE100", "EXERANY2", "DIFFWALK", "DIFFDRES", "DIFFALON", "VETERAN3",
              "ASTHMA3", "CVDINFR4", "CVDCRHD4", "CVDSTRK3", "TOLDHI3", "ADDEPEV3",
              "HAVARTH4", "CHCKDNY2", "MEDCOST1", "FLUSHOT7", "PNEUVAC4"]
    for c in yes_no:
        d[c] = d[c].map({1: 1, 2: 0})            # 7, 9, blank -> NaN

    d["is_female"] = (d["SEXVAR"] == 2).astype(int)

    # days variables
    for c in ["PHYSHLTH", "MENTHLTH"]:
        d[c] = d[c].replace({88: 0, 77: np.nan, 99: np.nan})

    # alcohol: drinking days per month
    a = d["ALCDAY4"]
    d["alc_days_month"] = np.select(
        [a == 888, a.between(101, 199), a.between(201, 299)],
        [0, (a - 100) * 30 / 7, a - 200],
        default=np.nan,
    )
    # SAS XPT stores a true 0 as a denormal (~5e-79); snap values that small to exactly 0
    drinks = d["_DRNKWK2"].replace(99900, np.nan) / 100
    d["alc_drinks_week"] = drinks.mask(drinks.abs() < 1e-6, 0)

    d["bmi"] = d["_BMI5"] / 100

    # categorical / ordinal codes with refusal values
    d["MARITAL"] = d["MARITAL"].replace({9: np.nan})
    d["EMPLOY1"] = d["EMPLOY1"].replace({9: np.nan})
    d["EDUCA"]   = d["EDUCA"].replace({9: np.nan})
    d["INCOME3"] = d["INCOME3"].replace({77: np.nan, 99: np.nan})
    d["GENHLTH"] = d["GENHLTH"].replace({7: np.nan, 9: np.nan})
    d["_SMOKER3"] = d["_SMOKER3"].replace({9: np.nan})
    d["SMOKDAY2"] = d["SMOKDAY2"].replace({7: np.nan, 9: np.nan})
    d["LASTSMK2"] = d["LASTSMK2"].replace({77: np.nan, 99: np.nan})
    d["USENOW3"]  = d["USENOW3"].replace({7: np.nan, 9: np.nan})
    d["DIABETE4"] = d["DIABETE4"].replace({7: np.nan, 9: np.nan})   # 1 yes, 2 yes (pregnancy only), 3 no, 4 prediabetes
    d["BPHIGH6"]  = d["BPHIGH6"].replace({7: np.nan, 9: np.nan})    # 1 yes, 2 yes (pregnancy only), 3 no, 4 borderline
    d["CHECKUP1"] = d["CHECKUP1"].replace({7: np.nan, 9: np.nan})   # 8 = never (kept as a real answer)

    d = d.drop(columns=["SEXVAR", "ALCDAY4", "_DRNKWK2", "_BMI5"])
    return d

clean = recode(df.drop(columns=[TARGET]))
print(clean.shape)
clean.head()

(269252, 41)


,_AGEG5YR,_AGE80,MARITAL,EMPLOY1,EDUCA,INCOME3,VETERAN3,_URBSTAT,_STATE,SMOKE100,...,CHCKDNY2,CHECKUP1,MEDCOST1,FLUSHOT7,PNEUVAC4,copd,is_female,alc_days_month,alc_drinks_week,bmi
0,13.0,80.0,1.0,7.0,5.0,NaN,0.0,1.0,1.0,0.0,...,0.0,2.0,0.0,0.0,0.0,0,1,0.0,0.00,30.47
1,13.0,80.0,2.0,7.0,5.0,NaN,0.0,1.0,1.0,0.0,...,0.0,2.0,0.0,1.0,1.0,0,1,0.0,0.00,28.56
2,13.0,80.0,3.0,7.0,4.0,2.0,0.0,1.0,1.0,1.0,...,0.0,1.0,1.0,1.0,1.0,0,1,0.0,0.00,22.31
3,12.0,78.0,1.0,7.0,5.0,NaN,0.0,1.0,1.0,0.0,...,0.0,3.0,0.0,1.0,1.0,0,1,0.0,0.00,27.44
4,12.0,76.0,3.0,8.0,5.0,7.0,0.0,1.0,1.0,0.0,...,0.0,1.0,0.0,0.0,1.0,0,1,2.0,0.47,25.85


## 6. Sanity checks and missing-value profile

Quick checks that the recoding behaved as intended (value ranges and category counts), followed by the share of missing values per column. Columns with a lot of missingness need an explicit strategy in the modelling notebooks.

In [8]:
print(clean[["_AGE80", "bmi", "PHYSHLTH", "MENTHLTH", "alc_days_month", "alc_drinks_week"]].describe().round(2))
print()
for c in ["GENHLTH", "_SMOKER3", "DIABETE4", "BPHIGH6", "_AGEG5YR"]:
    print(c, clean[c].value_counts(dropna=False).sort_index().to_dict())

          _AGE80        bmi   PHYSHLTH   MENTHLTH  alc_days_month  \
count  269252.00  248049.00  261616.00  264062.00       252517.00   
mean       67.12      28.50       5.16       3.39            5.14   
std         9.18       6.24       9.60       7.68            8.69   
min        50.00      12.02       0.00       0.00            0.00   
25%        60.00      24.33       0.00       0.00            0.00   
50%        67.00      27.44       0.00       0.00            0.00   
75%        75.00      31.57       5.00       2.00            5.00   
max        80.00      99.82      30.00      30.00           30.00   

       alc_drinks_week  
count        251138.00  
mean              2.54  
std               7.32  
min               0.00  
25%               0.00  
50%               0.00  
75%               2.00  
max             532.00  

GENHLTH {1.0: 33173, 2.0: 86385, 3.0: 89949, 4.0: 42925, 5.0: 15981, nan: 839}
_SMOKER3 {1.0: 19931, 2.0: 6956, 3.0: 83088, 4.0: 146445, nan: 12832}
DIA

In [9]:
miss = (clean.isna().mean() * 100).round(2).sort_values(ascending=False)
display(miss[miss > 0].to_frame("% missing"))

print("\nMissingness by COPD status (columns with the largest gap):")
gap = (clean.groupby("copd").apply(lambda g: g.isna().mean() * 100).T)
gap["diff"] = (gap[1] - gap[0]).abs()
display(gap.sort_values("diff", ascending=False).head(8).round(2))

,% missing
LASTSMK2,86.18
SMOKDAY2,59.16
INCOME3,20.37
PNEUVAC4,9.81
_BMI5CAT,7.87
bmi,7.87
alc_drinks_week,6.73
FLUSHOT7,6.35
alc_days_month,6.22
TOLDHI3,5.88



Missingness by COPD status (columns with the largest gap):


/tmp/ipykernel_3381/1403479541.py:5: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  gap = (clean.groupby("copd").apply(lambda g: g.isna().mean() * 100).T)


copd,0,1,diff
SMOKDAY2,62.97,26.87,36.10
LASTSMK2,87.03,78.97,8.06
CVDCRHD4,1.01,2.85,1.84
PNEUVAC4,9.99,8.23,1.76
PHYSHLTH,2.66,4.30,1.64
TOLDHI3,5.75,6.96,1.21
MENTHLTH,1.82,2.85,1.03
_BMI5CAT,7.98,7.02,0.96


## 7. Save the cleaned cohort

The saved Parquet file loads in seconds and keeps this notebook independent from the 1.2 GB raw file. Modelling notebooks should apply the stratified train/validation/test split themselves, with `SEED` fixed, and fit all imputers, scalers and encoders on the training set only to avoid leakage.

In [10]:
clean.to_parquet(OUT_PATH, index=False)
print("Saved", OUT_PATH, clean.shape)

Saved /content/drive/MyDrive/ml_project/brfss_copd_50plus.parquet (269252, 41)


In [11]:
import pandas as pd
df = pd.read_parquet("/content/drive/MyDrive/ml_project/brfss_copd_50plus.parquet")
print(df.shape)
print(df["copd"].value_counts())

(269252, 41)
copd
0    240820
1     28432
Name: count, dtype: int64


## 8. Next steps

1. Train/validation/test split (stratified, `SEED`).
2. Missing-value strategies compared (median/mode imputation, explicit "unknown" category, indicator columns).
3. Feature engineering (for example a cardiometabolic burden count, a functional-limitation count, smoking intensity and recency).
4. Traditional models (Logistic Regression, Random Forest, Gradient Boosting, SVM) with cross-validated tuning.
5. Keras MLPs (Functional API, `tf.data`) with imbalance handling.
6. Evaluation: learning curves, confusion matrices, ROC-AUC, precision/recall, threshold analysis, error analysis.

*Limitations to state in the report:* reported (not clinical) diagnosis; snapshot data, so it detects current cases rather than predicting future ones; undiagnosed COPD appears as a negative; sensitive attributes (income, sex, state) need an explicit fairness discussion.